# 04: Mobile Inverted Bottleneck Convolution Network (EfficientNet-B0)
**Benchmark:** BirdCLEF 2024 (15 Target Species Multiclass Acoustic Classification)  
**Methodology:** Zero Data Leakage, Pretrained MBConv Backbone, Compound Scaling, Activity-Aware Temporal Ensembling.


In [ ]:
# Cell 1: Environment Diagnostics & Setup
import os, sys, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.models as models
import librosa
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')


In [ ]:
# Cell 2: Ingest Artifacts & Configs
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'code' else Path('.').resolve()
DATA_DIR = PROJECT_ROOT / 'data_artifacts'
MODELS_DIR = PROJECT_ROOT / 'model_artifacts' / 'efficientnet_b0'
AUDIO_DIR = PROJECT_ROOT / 'sample_dataset' / 'audio'

with open(DATA_DIR / 'config' / 'preprocessing_config.json') as f: CONFIG = json.load(f)
with open(DATA_DIR / 'vocab' / 'species_mapping.json') as f: VOCAB = json.load(f)
print(f'Target Species: {VOCAB["num_classes"]}')


In [ ]:
# Cell 3: Build EfficientNet-B0 Architecture with Custom Bioacoustic Head
def build_efficientnet(num_classes=15):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_feat = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.35),
        nn.Linear(in_feat, 256, bias=False),
        nn.BatchNorm1d(256),
        nn.SiLU(inplace=True),
        nn.Dropout(p=0.25),
        nn.Linear(256, num_classes)
    )
    return m

model = build_efficientnet(VOCAB['num_classes']).to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'EfficientNet-B0 Trainable Parameters: {params:,}')


In [ ]:
# Cell 4: Inspect Training History & Evaluation Metrics
with open(MODELS_DIR / 'test_evaluation_report.json') as f: center_rep = json.load(f)
with open(MODELS_DIR / 'final_evaluation_report.json') as f: ens_rep = json.load(f)

print('--- CENTER CROP TEST METRICS ---')
print(f'Top-1 Test Accuracy : {center_rep["test_accuracy"]*100:.2f}%')
print(f'Top-3 Test Accuracy : {center_rep["test_accuracy_top3"]*100:.2f}%')
print(f'Test Macro F1       : {center_rep["test_macro_f1"]*100:.2f}%')

print('--- MULTI-WINDOW TEMPORAL ENSEMBLE METRICS ---')
print(f'Top-1 Test Accuracy : {ens_rep["test_accuracy_top1"]*100:.2f}%')
print(f'Top-3 Test Accuracy : {ens_rep["test_accuracy_top3"]*100:.2f}%')
print(f'Test Macro F1       : {ens_rep["test_macro_f1"]*100:.2f}%')


In [ ]:
# Cell 5: Display Training Curves & Confusion Matrix
from IPython.display import Image, display
display(Image(filename=str(MODELS_DIR / 'training_curves.png')))
display(Image(filename=str(MODELS_DIR / 'final_confusion_matrix.png')))
